**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/dvali-2025"], check=True)
    os.chdir("/content/encore/entries/dvali-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Dvali et al. (2025) from dandiset 001075

**Paper.** Dvali S, Seguin C, Betzel R, Leifer AM (2025). Diverging network architecture of the
*C. elegans* connectome and signaling network.
[doi:10.1103/6wgv-b9m6](https://doi.org/10.1103/6wgv-b9m6). The claims file was written from the
full text supplied with the attempt (its body is the arXiv preprint 2412.14498v2, with Methods
and Extended Data), and from the paper's code
([FunconnNetworkAnalysis](https://github.com/SophieDvali/FunconnNetworkAnalysis), commit 35e87ca).

**Dandiset.** [DANDI:001075](https://dandiarchive.org/dandiset/001075/0.240920.1434), version
0.240920.1434, the version the paper cites (Methods, The Networks; reference 88). The paper
reuses the signal propagation atlas of Randi et al. (2023). One later version exists,
0.240930.1859. It holds the same 223 files, and its 110 segmentation files differ: they add
`target_pumpprobe_id`, the ROI that each stimulation targeted, and split the green and red
signals into a series with missing frames as NaN and an interpolated one. The cited version does
not say which ROI was stimulated, so the stimulated-ROI identities, and only they, are read from
0.240930.1859 (listed under `inputs` in the record). The data check below shows the difference.

**What the dandiset holds and what it does not.** Per recording, the calcium signal of every
segmented ROI, the NeuroPAL labels, and the stimulation times and targets, for 110 recordings of
wild-type animals. It does not hold the signaling network itself, nor the q-values that define
it, nor the control recordings without stimulation that Randi et al.'s code uses to compute the
q-values (pumpprobe `Funatlas.get_ctrl_distributions`). Reading (a) of the claims file
(recompute the q-values from the dandiset) therefore cannot be run, and the signaling network is
taken under reading (b), from the wild-type atlas released with Randi et al. 2023
([worm-functional-connectivity](https://github.com/leiferlab/worm-functional-connectivity),
commit 81af01c), whose entries are NaN where q > 0.05. A link claim (`atlas-dataset-link`)
tests that this atlas matches the dandiset's recordings. The atlas holds no q-values, so the
networks at q < 0.01 and q <= 0.18 cannot be built.

**Outside inputs.** The anatomical network comes from the four EM datasets that Randi et al.'s
code combines (White 1986 adult and L4, Witvliet 2021 datasets 7 and 8), as stored in
[pumpprobe](https://github.com/leiferlab/pumpprobe) at commit 1dbc5e0. Neuron classes and roles
are the lists in the paper's code (compared with its source in the data check).

**Software.** The stochastic block model of the paper runs in graph-tool, which has no PyPI
distribution and cannot be installed in the entry's locked environment; every claim on
communities is blocked as `not_runnable` (PyPI holds no files for graph-tool; data check). The
paper prints the first-level memberships of both networks (Tables 1 and 2), but they are the
output of the model that cannot be run, so they are not typed in here as an input: the entry
computes every result from the data. The rich club null uses a degree-preserving rewiring
written here in place of graph-tool's `random_rewire`.

**Random seeds.** Each claim's generator is seeded with `zlib.crc32(b"dvali-2025/<claim id>")`;
results that depend on random draws are also given under the seed plus 1000, 2000, 3000 and 4000.

| Claim | Data | Status |
|---|---|---|
| atlas-dataset-link (link) | with inputs | tested |
| signaling-network-size | dataset (atlas, reading b) | tested |
| signaling-clustering-small-world | dataset (atlas) | tested |
| anatomy-network-properties | elsewhere | tested |
| triad-distribution | with inputs | tested |
| ed6-signaling-not-feedforward | with labels | tested |
| ed6-anatomy-feedforward | elsewhere | tested |
| fig4c-broadcaster-integrator | with labels | tested |
| ed9-sampling-not-confound | with labels | tested |
| fig5a-signaling-rich-club | dataset (atlas) | tested |
| fig5c-rich-clubs-disjoint | with inputs | tested |
| fig5b-degree-correlation | with inputs | tested |
| ed10-rich-club-cell-classes | with labels | tested |
| ed9e-degree-observations | dataset | tested |
| ave-most-connected-both | with inputs | tested |
| signaling-threshold-controls, bilateral-controls-signaling | dataset | blocked: q-values other than q < 0.05 not available |
| rich-club-controls | with inputs | blocked: q-values other than q < 0.05 not available; the one pair that can be built is shown |
| 21 claims on the stochastic block model communities and what depends on them | various | blocked: graph-tool not installable |

In [ ]:
import collections
import hashlib
import inspect
import io
import resource
import time
import warnings
import zlib

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from scipy import stats

import networks as nw

T0 = time.time()
KEY = "dvali-2025"
OFFSETS = [0, 1000, 2000, 3000, 4000]


def seed_of(claim_id, extra=0):
    """The claim's seed (protocol); `extra` is added for a second generator of the same claim."""
    return zlib.crc32(f"{KEY}/{claim_id}".encode()) + extra


warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## Data check

The archive's file lists of the two published versions, and what differs between them.

In [ ]:
from dandi.dandiapi import DandiAPIClient

client = DandiAPIClient()
versions = [v.identifier for v in client.get_dandiset(nw.DANDISET_ID).get_versions()]
print("versions on the archive:", versions)
files = {}
for v in (nw.DANDISET_VERSION, nw.STIM_ID_VERSION):
    files[v] = {a.path: a.size for a in client.get_dandiset(nw.DANDISET_ID, v).get_assets()}
v1, v2 = files[nw.DANDISET_VERSION], files[nw.STIM_ID_VERSION]
kinds = collections.Counter(p.split("desc-")[1] for p in v1)
print(f"{nw.DANDISET_VERSION}: {len(v1)} files, {sum(v1.values()) / 1e12:.2f} TB;", dict(kinds))
print(f"{nw.STIM_ID_VERSION}: {len(v2)} files; same paths: {set(v1) == set(v2)}")
changed = sorted(p for p in v1 if v1[p] != v2[p])
print("files whose size differs between versions:", len(changed),
      "of which segmentation files:", sum("segmentation" in p for p in changed))
sessions_im = {p.split("_desc")[0] for p in v1 if "desc-imaging" in p}
sessions_seg = {p.split("_desc")[0] for p in v1 if "desc-segmentation" in p}
print("recordings with raw imaging but no segmentation file:", sorted(sessions_im - sessions_seg))

Reading a file with pynwb. The files declare the `ndx-patterned-ogen` and `ndx-microscopy`
extensions in their cached specifications; the cell shows whether the installed pynwb builds
them. The analysis reads the files with h5py in any case.

In [ ]:
import pynwb

_, seg_assets = nw.list_segmentation_assets()
try:
    with pynwb.NWBHDF5IO(file=nw._open(seg_assets[0]), load_namespaces=True) as io_:
        io_.read()
    print("pynwb", pynwb.__version__, "reads", seg_assets[0].path)
except Exception as e:  # noqa: BLE001
    print("pynwb", pynwb.__version__, "cannot read", seg_assets[0].path)
    print(type(e).__name__, str(e)[-260:])

All 110 segmentation files, streamed (green signal, labels, ROI matching, stimulation table),
with the stimulated-ROI ids and the NaN-marked green signal of the later version.

In [ ]:
t = time.time()
sessions = nw.load_sessions(with_signal=True, workers=8)
print(f"loaded {len(sessions)} recordings in {time.time() - t:.0f} s; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
survey = pd.DataFrame([dict(
    path=s["path"], strain=s["strain"], genotype=s["genotype"], stage=s["stage"],
    start=s["start"], n_stim=len(s["stim_start"]), frames=s["signal_shape"][0],
    rois=s["signal_shape"][1], dt=float(np.median(np.diff(s["timestamps"]))),
    duration_s=float(s["timestamps"][-1] - s["timestamps"][0]),
    stim_interval_s=float(np.median(np.diff(s["stim_start"]))) if len(s["stim_start"]) > 1 else np.nan,
    stim_len_s=float(np.median(s["stim_stop"] - s["stim_start"])),
    rois_match_columns=bool(np.array_equal(s["signal_rois"], np.arange(s["signal_shape"][1]))),
    red_same_shape=s["red_shape"] == s["signal_shape"], red_same_clock=s["red_same_clock"],
) for s in sessions])
print(survey.groupby(["strain", "genotype"]).size())
print(survey.groupby("stage").size())
print(survey[["n_stim", "frames", "rois", "dt", "duration_s", "stim_interval_s", "stim_len_s"]]
      .describe().round(3))
print("signal columns are the ROIs in order in every file:", survey.rois_match_columns.all(),
      "; red signal has the green signal's shape in every file:", survey.red_same_shape.all(),
      "and its timestamps:", survey.red_same_clock.all())
print("total stimulations:", survey.n_stim.sum())

Duplicated recordings: identical green signals, or identical start times.

In [ ]:
digests = collections.defaultdict(list)
for s in sessions:
    digests[hashlib.sha1(np.ascontiguousarray(s["green"][:200]).tobytes()).hexdigest()].append(s["path"])
dups = [v for v in digests.values() if len(v) > 1]
print("recordings whose first 200 frames are identical to another's:", dups)
starts = survey.groupby("start").path.apply(list)
print("start times shared by more than one recording:", [v for v in starts if len(v) > 1])
# Same subject ids across files: each file is its own subject (sub-<n>).
print("distinct subjects:", survey.path.str.split("/").str[0].nunique())

The green signal of the cited version against the later version. The later version's
`BaseGreenSignal` keeps the frames that could not be measured as NaN.

In [ ]:
rows = []
for s in sessions:
    base = s["later"].pop("base")
    fin = np.isfinite(base)
    rows.append(dict(path=s["path"], cited_nan=float(np.isnan(s["green"]).mean()),
                     later_nan=float(1 - fin.mean()),
                     equal_where_measured=bool(np.allclose(s["green"][fin], base[fin])),
                     same_shape=base.shape == s["green"].shape))
    s["missing"] = ~fin
vd = pd.DataFrame(rows)
print("same shape in every file:", vd.same_shape.all())
print("cited GreenSignal equals the later BaseGreenSignal wherever the latter is measured:",
      vd.equal_where_measured.all())
print("fraction of NaN in the cited GreenSignal: max", vd.cited_nan.max())
print("fraction of frames x ROIs that the later version marks as not measured "
      "(interpolated, unmarked, in the cited version):")
print(vd.later_nan.describe().round(3))

Labels. NeuroPAL labels are matched to the 188 head neuron names of the atlas by exact name.
Labels that name a bilateral class without its side, numbers and free text are counted apart.

In [ ]:
names, scalar_atlas = nw.load_atlas()
N = len(names)
name_set = set(names)
idx = {n: i for i, n in enumerate(names)}
lab_kinds = collections.Counter()
dup_in_rec = 0
examples = collections.defaultdict(collections.Counter)
for s in sessions:
    labs = [x for x in s["roi_label"] if x]
    c = collections.Counter(labs)
    dup_in_rec += sum(v - 1 for k, v in c.items() if k in name_set and v > 1)
    for x in labs:
        if x in name_set:
            k = "head neuron name"
        elif any(x + side in name_set for side in ("L", "R")):
            k = "class without side"
        elif x.isdigit():
            k = "number"
        else:
            k = "other"
        lab_kinds[k] += 1
        examples[k][x] += 1
print("labels of ROIs in the activity recordings:", dict(lab_kinds))
for k in ("class without side", "number", "other"):
    print(f"  {k}: most common", examples[k].most_common(8))
print("ROIs whose head-neuron label repeats another ROI's in the same recording:", dup_in_rec)
print("atlas neurons:", N, "; all names in the paper's class lists:",
      set(names) == set(nw.SENSORY) | set(nw.INTER) | set(nw.MOTOR))

What the outside sources hold. Every attribute of the atlas pickle, and every file other than
code in the three repositories at their pinned commits, to show whether any of them holds
q-values or an atlas not thresholded at q < 0.05. The TSV of worm-functional-connectivity is an
export of one scalar per pair (`scripts/export_atlas_scalar.py`). Also the number of files of
graph-tool on PyPI, from which the registry installs packages.

In [ ]:
print("atlas pickle attributes (type, shape):")
for k, v in nw.atlas_attributes().items():
    print(f"  {k}: {v}")
code_ext = (".py", ".pyc", ".cpp", ".hpp", ".tcc", ".h", ".o", ".so", ".sh", ".iml", ".xml")
for repo, commit in ((nw.WFC_REPO, nw.WFC_COMMIT), (nw.PUMPPROBE_REPO, nw.PUMPPROBE_COMMIT),
                     (nw.FNA_REPO, nw.FNA_COMMIT)):
    files_ = nw.repo_files(repo, commit)
    data_ = {k: v for k, v in files_.items() if not k.endswith(code_ext) and not k.startswith("build/")}
    print(f"{repo} at {commit[:7]}: {len(files_)} files, {len(data_)} not code (build/ left out):")
    for k, v in data_.items():
        print(f"  {k} ({v} bytes)")
tsv = pd.read_csv(io.BytesIO(nw._raw(nw.WFC_REPO, nw.WFC_COMMIT,
                                    "scripts/leifer_funconn_wildtype_2022-04-01.tsv")),
                  sep="\t", header=None, names=["from", "to", "value"])
print(f"worm-functional-connectivity TSV: {len(tsv)} rows, {tsv['from'].nunique()} neurons, "
      f"one value per pair, {tsv.value.nunique()} distinct values, range {tsv.value.min()} to "
      f"{tsv.value.max()}")
print("graph-tool distribution files on PyPI, all releases:", nw.pypi_files("graph-tool"))

The neuron lists of the helper against the lists written in the paper's code, parsed from its
source at the pinned commit, and the code's `swapping_edges` (SIM_doubleedge.py), which
`nw.swapping_edges` ports. Extended Data Figure 5 shows the class and role annotations as an
image; its neuron names are not in the text of the paper, so the lists can be compared only with
the code.

In [ ]:
helper_lists = {"sensory": nw.SENSORY, "inter": nw.INTER, "motor": nw.MOTOR, "pharynx": nw.PHARYNX,
                "Amphid": nw.AMPHID}
for (fname, var), lst in nw.code_lists().items():
    h = helper_lists[var]
    print(f"{fname} {var}: {len(lst)} names; same as the helper's list: {lst == h}; "
          f"only in the code: {sorted(set(lst) - set(h))}; only in the helper: {sorted(set(h) - set(lst))}")
print("class lists disjoint:", not (set(nw.SENSORY) & set(nw.INTER) or set(nw.SENSORY) & set(nw.MOTOR)
                                    or set(nw.INTER) & set(nw.MOTOR)))
print("pharynx and amphid neurons among the atlas's 188:", len(set(nw.PHARYNX) & name_set),
      len(set(nw.AMPHID) & name_set))
print("--- swapping_edges in SIM_doubleedge.py:")
print(nw.code_function_source("SIM_doubleedge.py", "swapping_edges"))
print("--- the port in the helper:")
print(inspect.getsource(nw.swapping_edges))

Stimulated ROIs (from the later version), and the clock. If the stimulation times share the
clock of the signals, the targeted ROI responds after the stimulation onset and not 15 s later
(half-way between stimulations, which come every 30 s). A shift of the times that does not
change the response would mean the times are on another clock.

In [ ]:
n_ids = n_lab = n_head = 0
own = {0.0: [], 15.0: [], -15.0: []}
per_session = []
for s in sessions:
    ids = s["later"]["ids"]
    assert np.allclose(s["later"]["start"], s["stim_start"])
    labs = nw.stimulated_labels(s)
    n_ids += int(np.isfinite(ids).sum())
    n_lab += sum(1 for x in labs if x)
    n_head += sum(1 for x in labs if x in name_set)
    r = {}
    for sh in own:
        R = nw.responses(s, shift_s=sh)
        vals = [R[k, int(i)] for k, i in enumerate(ids) if np.isfinite(i)]
        own[sh] += vals
        r[sh] = np.nanmedian(vals) if len(vals) else np.nan
    per_session.append(r)
print(f"stimulations {survey.n_stim.sum()}; with a targeted ROI id {n_ids}; "
      f"the ROI has a label {n_lab}; the label is a head neuron name {n_head}")
for sh, v in own.items():
    print(f"targeted ROI, dF/F over 30 s after onset shifted by {sh:+.0f} s: median "
          f"{np.nanmedian(v):.3f}, n = {np.isfinite(v).sum()}")
ps = pd.DataFrame(per_session)
ok = ps[[0.0, 15.0]].dropna()
print(f"recordings where the targeted ROIs respond more at the true onset than 15 s later: "
      f"{(ok[0.0] > ok[15.0]).sum()} of {len(ok)}; "
      f"recordings with no targeted ROI id: {ps[0.0].isna().sum()}")
w = stats.wilcoxon(np.array(own[0.0]), np.array(own[15.0]), nan_policy="omit")
print(f"paired Wilcoxon, true onset against +15 s: p = {w.pvalue:.2g}")

**Checks of the description against the data.** The archive lists 113 recordings with raw
imaging and 110 with segmentation files; the three without (sub-20, sub-23, sub-33) cannot be
used. All 110 are wild type (`genotype` WT); 108 give the strain AML462 and two (sub-24, sub-25)
give "none". The volume rate is one per 0.501 s in every file, stimulations come every 30.8 s
(median) and last 0.5 s, as the stimulation table's description says (every 30 s, 500 ms). No
recording is deposited twice. The stimulation times share the clock of the signals: the targeted
ROI responds after the true onset (median dF/F 0.204) and not 15 s later (-0.092), in 108 of
the 108 recordings with targeted ROI ids. Two things in the files differ from what they say. The
cited version's `GreenSignal` is described as the measured fluorescence, but the later version
shows that 17% to 68% of its values per file (mean 42%) are frames that were not measured and
were filled by interpolation, with nothing in the cited files to mark them. And the cited
version does not say which ROI each stimulation targeted; the later version does, for 2815 of
the 5614 stimulations. Of the labels of activity ROIs, 6659 are head neuron names, 487 name a
bilateral class without its side, and 307 are numbers or other text ("target", "AMso",
"AWCON"), with no description of what they mean. pynwb 4.2.0 cannot build the files. Of
the outside sources, the atlas pickle holds the names, the fitted kernels (`ec`) and the scalar
atlas thresholded at q < 0.05, and neither its repository nor pumpprobe nor the paper's code
holds a q-value matrix or an unthresholded atlas (the TSV is a scalar export of 300 neurons).
The class and role lists of the helper are the same as those written in the paper's code.

## atlas-dataset-link

> The signaling network of the released atlas (Randi et al. 2023, wild type, q < 0.05) can be
> matched to the dandiset's recordings: its neurons are named as the dandiset's labels, its
> edges are pairs the recordings measured, and its response amplitudes agree with the
> responses in the recordings.

Records are matched by neuron name. For each stimulation whose targeted ROI carries a head
neuron label (neuron j), every other labeled head-neuron ROI i of the same recording is an
occurrence of the pair (j to i), and its response is the mean dF/F over the 30 s after onset
against the 10 s before. The atlas value is `scalar_atlas[i, j]`.

In [ ]:
A = nw.signaling_adjacency(scalar_atlas)          # A[j, i]: stimulated j to responding i
occ = np.zeros((N, N), int)                       # occ[i, j]: i observed when j stimulated
resp_sum = np.zeros((N, N))
resp_n = np.zeros((N, N), int)
stim_count = np.zeros(N, int)
obs_count = np.zeros(N, int)                      # recordings with an activity ROI labeled i
obs_np = np.zeros(N, int)                         # recordings with a NeuroPAL ROI labeled i
for s in sessions:
    heads = [(c, idx[l]) for c, l in enumerate(s["roi_label"]) if l in name_set]
    for a in {a for _, a in heads}:
        obs_count[a] += 1
    for a in {idx[l] for l in s["np_labels"] if l in name_set}:
        obs_np[a] += 1
    R = nw.responses(s)
    for k, (lab, roi) in enumerate(zip(nw.stimulated_labels(s), s["later"]["ids"])):
        if lab not in name_set:
            continue
        j = idx[lab]
        stim_count[j] += 1
        seen = set()
        for c, a in heads:
            if c == int(roi) or a == j or a in seen:
                continue
            seen.add(a)
            occ[a, j] += 1
            if np.isfinite(R[k, c]):
                resp_sum[a, j] += R[k, c]
                resp_n[a, j] += 1
mean_resp = np.where(resp_n > 0, resp_sum / np.maximum(resp_n, 1), np.nan)
edge = A.T.astype(bool)                           # edge[i, j], the atlas orientation
print("atlas neurons named as dandiset labels:", sum(n in {l for s in sessions for l in s['roi_label']}
                                                    for n in names), "of", N)
print(f"atlas edges {edge.sum()}; with at least one occurrence in the dandiset: "
      f"{(occ[edge] > 0).sum()} ({(occ[edge] > 0).mean():.3f})")
print(f"same check with the atlas transposed (edges read as responding to stimulated): "
      f"{(occ[edge.T] > 0).mean():.3f}")
print("occurrences per atlas edge: median", np.median(occ[edge]), "range", occ[edge].min(),
      "to", occ[edge].max())
obs_pairs = (occ > 0) & ~np.eye(N, dtype=bool)
print("pairs the dandiset measured at least once:", obs_pairs.sum(), "; of them atlas edges:",
      (obs_pairs & edge).sum())
m = edge & np.isfinite(mean_resp)
rho = stats.spearmanr(scalar_atlas[m], mean_resp[m])
print(f"atlas amplitude against the dandiset's mean dF/F over {m.sum()} atlas edges: "
      f"Spearman rho = {rho.statistic:.3f}, p = {rho.pvalue:.2g}")
ne = obs_pairs & ~edge & np.isfinite(mean_resp)
mw = stats.mannwhitneyu(np.abs(mean_resp[m]), np.abs(mean_resp[ne]), alternative="greater")
print(f"|mean dF/F|: atlas edges median {np.median(np.abs(mean_resp[m])):.3f} (n = {m.sum()}), "
      f"measured non-edges median {np.median(np.abs(mean_resp[ne])):.3f} (n = {ne.sum()}); "
      f"Mann-Whitney (edges greater) p = {mw.pvalue:.2g}")
# Orientation: out-degree should follow the number of stimulations, in-degree the observations.
outdeg, indeg = A.sum(1), A.sum(0)
for lab_, x, y in (("out-degree vs stimulations", outdeg, stim_count),
                   ("in-degree vs stimulations", indeg, stim_count),
                   ("in-degree vs observations", indeg, obs_count),
                   ("out-degree vs observations", outdeg, obs_count)):
    r_ = stats.spearmanr(x, y)
    print(f"{lab_}: Spearman {r_.statistic:.3f}")
asym = (edge & ~edge.T & (occ > 0) & (occ.T > 0) & np.isfinite(mean_resp)
        & np.isfinite(mean_resp.T))
fr = (np.abs(mean_resp[asym]) > np.abs(mean_resp.T[asym])).mean()
print(f"atlas edges j to i without the reverse edge, both directions measured ({asym.sum()} "
      f"pairs): |response of i to j| exceeds |response of j to i| in {fr:.3f}")
print("neurons never stimulated with a label in the dandiset:", (stim_count == 0).sum(),
      "; their atlas out-degree total:", outdeg[stim_count == 0].sum())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(scalar_atlas[m], mean_resp[m], s=6)
ax[0].set_xlabel("atlas amplitude")
ax[0].set_ylabel("dandiset mean dF/F (0 to 30 s)")
ax[0].set_xlim(-1, 3)
ax[0].set_ylim(-1, 3)
ax[1].hist([np.abs(mean_resp[m]), np.abs(mean_resp[ne])], bins=np.linspace(0, 2, 41),
           density=True, label=["atlas edges", "measured non-edges"])
ax[1].set_xlabel("|mean dF/F|")
ax[1].legend()
plt.tight_layout()
plt.show()
print("points outside the left axes:", int(((scalar_atlas[m] < -1) | (scalar_atlas[m] > 3)
                                         | (mean_resp[m] < -1) | (mean_resp[m] > 3)).sum()),
      "; values above 2 outside the right histogram:",
      int((np.abs(mean_resp[m]) > 2).sum() + (np.abs(mean_resp[ne]) > 2).sum()))

**Result: partially reproduced.** The amplitude tests hold: over the 1020 atlas edges the dandiset
measured, the atlas amplitude ranks with the recordings' mean dF/F (Spearman rho 0.524), and
atlas edges respond more than measured non-edges (median |dF/F| 0.074 against 0.042). The
coverage tests fail: 184 of the 188 atlas neurons are among the activity ROIs' labels, and 1020
of the 1154 atlas edges (0.884) have an occurrence, below the 95% asked. The dandiset gives a
targeted ROI for 2815 of its 5614 stimulations, 1813 of them a head neuron label, and 23 neurons
never appear as a labeled target although the atlas gives them 46 out-edges; the atlas rests on
stimulation identities that the dandiset does not hold in full. Asymmetric edges support the
stimulated-to-responding orientation (0.626 respond more in the atlas direction); the degree
correlations barely separate the two orientations.

## signaling-network-size

> The binarized signaling network of the 188 head neurons, with an edge for each pair whose
> q-value is below 0.05, has 1151 edges and a density of 0.033.

Reading (a), q-values recomputed from the dandiset, cannot be run: Randi et al.'s q-values
test each pair's responses against a null distribution from control recordings without
stimulation (pumpprobe `Funatlas.get_ctrl_distributions`, `kstest` against `ctrl_dff` and
`ctrl_sd`), and those recordings are not in the dandiset (all 110 recordings have
stimulations, see the data check). Reading (b) uses the released atlas.

In [ ]:
n_edges = int(A.sum())
dens = nw.density(A)
print(f"reading (b): {N} neurons, {n_edges} edges (expected 1151 +/- 115), density {dens:.4f} "
      f"(expected 0.033 +/- 0.0033)")
print("self-edges dropped:", int(np.isfinite(np.diag(scalar_atlas)).sum()))
print("reading (a): not run, the control recordings are not in the dandiset")

**Result: reproduced under reading (b).** The atlas has 1154 edges among the 188 head neurons,
density 0.0328, both within tolerance; 3 edges more than the paper's 1151.

## signaling-clustering-small-world

> The signaling network has an average clustering coefficient of 0.13 and is small world,
> with a small-world coefficient of 4.76.

As in motifs.py: 100 directed Erdos-Renyi graphs at density 0.033.

In [ ]:
cid = "signaling-clustering-small-world"
sw_runs = {}
for off in OFFSETS:
    sw_runs[off] = nw.small_world(A, 0.033, np.random.default_rng(seed_of(cid) + off))
sw = pd.DataFrame(sw_runs).T
sw.index.name = "seed offset"
print(sw.round(4))
print(f"claim's seed: C = {sw.loc[0, 'C']:.4f} (expected 0.13 +/- 0.013), sigma = "
      f"{sw.loc[0, 'sigma']:.3f} (expected 4.76 +/- 0.48); sigma over five seeds "
      f"{sw.sigma.min():.3f} to {sw.sigma.max():.3f}")

**Result: reproduced.** The clustering coefficient is 0.135 and sigma about 4.6, above 1 and
within tolerance under all five seeds.

## anatomy-network-properties (elsewhere)

> The binarized anatomical network has 3250 edges, a density of 0.092, an average clustering
> coefficient of 0.3 and a small-world coefficient of 2.52.

The four EM datasets are not named in the paper. The main reading uses the four sources of
Randi et al.'s code (White 1986 whole adult, White 1986 L4, Witvliet 2021 datasets 7 and 8),
with gap junctions in both directions. Variants: White 1986 adult N2U (`white_1986_A`) in place
of the whole-animal White adult, and gap junctions only in the direction listed.

In [ ]:
cid = "anatomy-network-properties"
anat_src = nw.load_anatomy(names)
B = nw.anatomy_adjacency(anat_src, "any")
rows = {}
for off in OFFSETS:
    rows[off] = nw.small_world(B, 0.092, np.random.default_rng(seed_of(cid) + off))
swa = pd.DataFrame(rows).T
swa.index.name = "seed offset"
print(f"main reading: {int(B.sum())} edges (expected 3250 +/- 325), density {nw.density(B):.4f} "
      f"(0.092 +/- 0.0092)")
print(swa.round(4))
variants = {
    "White A (N2U) for White whole": nw.load_anatomy(
        names, ["aconnectome_white_1986_A.csv"] + nw.ANATOMY_SOURCES[1:]),
    "gap junctions one direction": nw.load_anatomy(names, symmetrize_gaps=False),
}
for k, src_ in variants.items():
    Bv = nw.anatomy_adjacency(src_, "any")
    swv = nw.small_world(Bv, 0.092, np.random.default_rng(seed_of(cid)))
    print(f"variant {k}: {int(Bv.sum())} edges, density {nw.density(Bv):.4f}, "
          f"C {swv['C']:.4f}, sigma {swv['sigma']:.3f}")

**Result: reproduced, resting on the choice of the four EM datasets.** With the sources of
Randi et al.'s code the network has 3223 edges, density 0.092, clustering 0.306 and sigma
about 2.5, all within tolerance. The paper does not name its four datasets; under the variants
the outcome changes: with the White N2U adult in place of the whole-animal White adult, all four values fall outside
tolerance (2893 edges, density 0.0823 below 0.0828, clustering 0.233, sigma 2.93), and with gap
junctions in one direction sigma (2.21) does.

## triad-distribution

> The signaling and anatomical networks differ in their distribution of the 13 triad motifs,
> each compared with Erdos-Renyi graphs of matched density.

Counts of the 13 connected triad types (networkx `triadic_census`, the same counts as
`triads_by_type`) in each network and averaged over five Erdos-Renyi graphs at each density
(Extended Data Figure 1 legend; motifs_ER.py). The chi-square test of the 13 x 2 table is the
claims file's choice.

In [ ]:
cid = "triad-distribution"
tri = pd.DataFrame({"signaling": nw.triad_counts(A), "anatomy": nw.triad_counts(B)})
rng = np.random.default_rng(seed_of(cid))
for lab_, p in (("ER 0.033", 0.033), ("ER 0.092", 0.092)):
    ers = [nw.triad_counts(nx.to_numpy_array(nx.erdos_renyi_graph(N, p, directed=True,
                                                                     seed=int(rng.integers(2**31)))))
           for _ in range(5)]
    tri[lab_ + " mean of 5"] = pd.DataFrame(ers).mean().round(1)
tri["signaling / ER"] = (tri["signaling"] / tri["ER 0.033 mean of 5"]).round(2)
tri["anatomy / ER"] = (tri["anatomy"] / tri["ER 0.092 mean of 5"]).round(2)
print(tri)
chi = stats.chi2_contingency(tri[["signaling", "anatomy"]].to_numpy())
print(f"chi-square on the proportions of the 13 types: chi2 = {chi.statistic:.1f}, "
      f"dof = {chi.dof}, p = {chi.pvalue:.3g}")
for lab_, src_ in variants.items():
    tv = pd.Series(nw.triad_counts(nw.anatomy_adjacency(src_, "any")))
    cv = stats.chi2_contingency(np.c_[tri["signaling"].to_numpy(), tv.to_numpy()])
    print(f"variant {lab_}: chi2 = {cv.statistic:.1f}, p = {cv.pvalue:.3g}")
prop = tri[["signaling", "anatomy"]] / tri[["signaling", "anatomy"]].sum()
print("proportions:")
print(prop.round(3))

**Result: reproduced.** The proportions of the 13 triad types differ between the two networks
(chi2 = 6008.6 on 12 degrees of freedom, p printed as 0), also under both anatomical variants.
Relative to Erdos-Renyi graphs of matched density, the signaling network keeps the open two-edge
triads near chance (021D 0.99, 021U 0.85 times the ER count) while anatomy has fewer than chance
(0.29, 0.38), and anatomy has 688 times the ER count of fully connected triads (300).

## ed6-signaling-not-feedforward

> The signaling network is not appreciably feed-forward along the sensory to inter- to motor
> neuron path compared with shuffle controls.

For each leg, log(N forward / N backward) against (b) 1000 shuffles of the class labels keeping
class sizes (SIM.py) and (c) 1000 networks rewired by the edge swap of SIM_doubleedge.py
(2000 swaps in at most 4000 counted attempts, as its line 237 calls it), by a
two-sided t-test of the observed value against the null values. Classes are the code's lists.
The code's other measure, the log ratio of the fractions of each class's out-edges, is a variant.

In [ ]:
cls = np.array(["S" if n in nw.SENSORY else "I" if n in nw.INTER else "M" for n in names])
print("class sizes:", collections.Counter(cls))


def feedforward_test(adj, claim_id, n_null=1000, offset=0, nswap=2000, max_tries=4000):
    obs = nw.feedforward_measures(nw.class_flow(adj, cls))
    rng_b = np.random.default_rng(seed_of(claim_id) + offset)
    rng_c = np.random.default_rng(seed_of(claim_id, 1) + offset)
    null_b, null_c, n_swaps = [], [], []
    for _ in range(n_null):
        null_b.append(nw.feedforward_measures(nw.class_flow(adj, rng_b.permutation(cls))))
        sw, n_sw = nw.swapping_edges(adj, rng_c, nswap, max_tries)
        n_swaps.append(n_sw)
        null_c.append(nw.feedforward_measures(nw.class_flow(sw, cls)))
    rows = []
    for ctrl, null in (("label shuffle", pd.DataFrame(null_b)), ("edge swap", pd.DataFrame(null_c))):
        for meas in ("count", "frac"):
            for leg in ("SI", "IM", "SM"):
                k = f"{meas}_{leg}"
                v = null[k].replace([np.inf, -np.inf], np.nan).dropna()
                t_ = stats.ttest_ind(obs[k], v.to_numpy(), alternative="two-sided")
                rows.append(dict(control=ctrl, measure=meas, leg=leg, observed=obs[k],
                                 null_mean=v.mean(), null_sd=v.std(), n_null=len(v),
                                 z=(obs[k] - v.mean()) / v.std(), p=t_.pvalue,
                                 above=bool(t_.pvalue < 0.05 and obs[k] > v.mean())))
    return pd.DataFrame(rows), nw.class_flow(adj, cls), np.array(n_swaps)


def swap_summary(n_swaps, nswap):
    return (f"swaps per edge-swap null network: min {n_swaps.min()}, median {np.median(n_swaps):.0f}, "
            f"max {n_swaps.max()}; {int((n_swaps < nswap).sum())} of {len(n_swaps)} stopped at max_tries "
            f"before {nswap}")

In [ ]:
cid = "ed6-signaling-not-feedforward"
t = time.time()
ff_sig, flow_sig, nsw_sig = feedforward_test(A, cid)
print("edge counts between classes (row source, column target):")
print(pd.DataFrame([[flow_sig[s + u] for u in "SIM"] for s in "SIM"], index=list("SIM"),
                   columns=list("SIM")))
print(ff_sig.round(4).to_string())
print(swap_summary(nsw_sig, 2000), f"(the network has {int(A.sum())} edges)")
main = ff_sig[ff_sig.measure == "count"]
print("legs significantly above the null (count measure):",
      main[main.above][["control", "leg"]].values.tolist())
print(f"({time.time() - t:.0f} s)")
seed_rows = []
for off in OFFSETS[1:]:
    r_, _, nsw_ = feedforward_test(A, cid, offset=off)
    rc = r_[r_.measure == "count"]
    rf = r_[r_.measure == "frac"]
    seed_rows.append(dict(offset=off, count_legs_above=int(rc.above.sum()),
                          count_max_p=rc.p.max(), count_min_p=rc.p.min(),
                          frac_legs_above=int(rf.above.sum()), frac_min_p=rf.p.min(),
                          min_swaps=nsw_.min()))
print("other seeds (count and fraction measures):")
print(pd.DataFrame(seed_rows).round(6).to_string(index=False))

**Result: reproduced.** No leg is significantly above either null for the count measure
(label shuffle p = 0.0092, 0.68, 0.055; edge swap p = 0.062, 0.55, 0.74), nor for the code's
fraction measure; under each of the four other seeds no leg is above for either measure (count
measure smallest p 0.0063 to 0.0148, fraction measure 0.0011 to 0.0020, all for values below the
null). Every edge-swap null network received its 2000 swaps. The observed values are all
negative: the sensory to inter leg is significantly below the label-shuffle null (z = -2.61), more
backward than forward.

## ed6-anatomy-feedforward (elsewhere)

> Anatomical synaptic edges are enriched in a feed-forward direction from sensory to inter- to
> motor neurons.

The claims file gives no criterion for this claim; the paper's test is the one of
ed6-signaling-not-feedforward (Extended Data Figure 6 b, c; SIM.py and SIM_doubleedge.py run
both networks the same way), applied to the anatomical network.
The edge swap makes 6000 swaps in at most 9000 counted attempts, as SIM_doubleedge.py (line 493)
calls it for the anatomical network.

In [ ]:
cid = "ed6-anatomy-feedforward"
ff_an, flow_an, nsw_an = feedforward_test(B, cid, nswap=6000, max_tries=9000)
print(pd.DataFrame([[flow_an[s + u] for u in "SIM"] for s in "SIM"], index=list("SIM"),
                   columns=list("SIM")))
print(ff_an.round(4).to_string())
print(swap_summary(nsw_an, 6000), f"(the network has {int(B.sum())} edges)")
seed_rows = []
for off in OFFSETS[1:]:
    r_, _, nsw_ = feedforward_test(B, cid, offset=off, nswap=6000, max_tries=9000)
    rc = r_[r_.measure == "count"]
    seed_rows.append(dict(offset=off, legs_above=int(rc.above.sum()),
                          legs_above_list=rc[rc.above].apply(lambda x: f"{x.control}:{x.leg}", axis=1).tolist(),
                          min_swaps=nsw_.min()))
print("other seeds (count measure):")
print(pd.DataFrame(seed_rows).to_string())
for lab_, src_ in variants.items():
    r_, _, nsw_ = feedforward_test(nw.anatomy_adjacency(src_, "any"), cid, nswap=6000, max_tries=9000)
    rc = r_[r_.measure == "count"]
    print(f"variant {lab_}: legs above the null:",
          rc[rc.above].apply(lambda x: f"{x.control}:{x.leg}", axis=1).tolist())
    print("   ", rc[["control", "leg", "z", "p"]].round(4).values.tolist())
    print("   ", swap_summary(nsw_, 6000))

**Result: partially reproduced.** Against shuffled class labels all three legs are
significantly feed-forward (z = 4.78, 2.30 and 10.12). Against the edge-swap control, with 6000
swaps per null network as the code makes for the anatomical network, only the sensory to motor
leg is (z = 8.31); sensory to inter (z = 0.45, p = 0.65) and inter to motor (z = 0.60, p = 0.55)
are not. The same legs come out under all five seeds and with gap junctions in one direction;
with the White N2U adult the inter to motor leg is not above the label shuffle either (z = 1.48,
p = 0.14). The criterion, every leg above both nulls for reproduced and some for partial, is the
author's, as the claims file gives none.

## fig4c-broadcaster-integrator

> In the signaling network pharyngeal neurons act as broadcasters, with more outgoing than
> incoming edges, and amphid sensory neurons have more incoming than outgoing edges, each
> compared with shuffled role assignments.

As in broadcaster_integrator.py: edges into each group from outside over edges out of it to
outside, against 1000 random disjoint pairs of groups of 20 and 24 neurons; one-sided t-tests
(pharynx lower than the null, amphid higher).

In [ ]:
cid = "fig4c-broadcaster-integrator"
ph = np.isin(names, nw.PHARYNX)
am = np.isin(names, nw.AMPHID)
print("group sizes in the atlas:", ph.sum(), am.sum())


def broadcaster(adj, offset=0, n_null=1000):
    rng_ = np.random.default_rng(seed_of(cid) + offset)
    ip, op = nw.boundary_ratio(adj, ph)
    ia, oa = nw.boundary_ratio(adj, am)
    null_p, null_a = [], []
    for _ in range(n_null):
        perm = rng_.permutation(N)
        gp = np.zeros(N, bool)
        gp[perm[:ph.sum()]] = True
        ga = np.zeros(N, bool)
        ga[perm[ph.sum():ph.sum() + am.sum()]] = True
        i_, o_ = nw.boundary_ratio(adj, gp)
        null_p.append(i_ / o_ if o_ else np.nan)
        i_, o_ = nw.boundary_ratio(adj, ga)
        null_a.append(i_ / o_ if o_ else np.nan)
    null_p, null_a = np.array(null_p), np.array(null_a)
    pp_ = stats.ttest_ind(ip / op, null_p[np.isfinite(null_p)], alternative="less").pvalue
    pa_ = stats.ttest_ind(ia / oa, null_a[np.isfinite(null_a)], alternative="greater").pvalue
    return dict(ph_in=ip, ph_out=op, ph_ratio=ip / op, ph_null_mean=np.nanmean(null_p),
                ph_null_sd=np.nanstd(null_p), ph_p=pp_, am_in=ia, am_out=oa, am_ratio=ia / oa,
                am_null_mean=np.nanmean(null_a), am_null_sd=np.nanstd(null_a), am_p=pa_,
                null_dropped=int((~np.isfinite(null_p)).sum() + (~np.isfinite(null_a)).sum()))


bi = pd.DataFrame({off: broadcaster(A, off) for off in OFFSETS}).T
bi.index.name = "seed offset"
print(bi.T.to_string())

**Result: partially reproduced.** Amphid sensory neurons have more incoming than outgoing
boundary edges (160 against 100, ratio 1.60 against a null mean of 1.03, p = 0.0046). Pharyngeal
neurons have more outgoing than incoming boundary edges (60 against 40, ratio 0.67 against a
null mean of 1.02), but the one-sided t-test gives p = 0.073, above 0.05; under the other four
seeds p is 0.051 to 0.063, so the outcome does not depend on the seed.

## ed9-sampling-not-confound

> Pharyngeal neurons were not stimulated more often than other neurons (Kolmogorov-Smirnov
> p = 0.9), and amphid neurons were not observed more often (p = 0.38).

Per neuron, the number of stimulations is the number of stimulation events whose targeted ROI
carries its label (targeted ROI ids from version 0.240930.1859), and the number of observations
the number of recordings in which an activity ROI carries its label (as pumpprobe
`get_times_i_observed` counts them). Two-sample KS test of the group against all other head
neurons (broadcaster_integrator.py). Variant: observations counted from the NeuroPAL labels.

In [ ]:
ks_p = stats.ks_2samp(stim_count[~ph], stim_count[ph])
ks_a = stats.ks_2samp(obs_count[~am], obs_count[am])
ks_av = stats.ks_2samp(obs_np[~am], obs_np[am])
print(f"stimulations: pharynx median {np.median(stim_count[ph])} (range {stim_count[ph].min()} to "
      f"{stim_count[ph].max()}), others median {np.median(stim_count[~ph])}; KS D = "
      f"{ks_p.statistic:.3f}, p = {ks_p.pvalue:.3g}")
print(f"observations: amphid median {np.median(obs_count[am])} (range {obs_count[am].min()} to "
      f"{obs_count[am].max()}), others median {np.median(obs_count[~am])}; KS D = "
      f"{ks_a.statistic:.3f}, p = {ks_a.pvalue:.3g}")
print(f"variant, NeuroPAL observations: amphid median {np.median(obs_np[am])}, others "
      f"{np.median(obs_np[~am])}; KS p = {ks_av.pvalue:.3g}")
print("total stimulations counted:", stim_count.sum(), "; neurons with none:", (stim_count == 0).sum())
for g, nm in ((ph, "pharynx"), (am, "amphid")):
    print(f"{nm} neurons with no stimulation: {(stim_count[g] == 0).sum()} of {g.sum()}")

**Result: reproduced.** Pharyngeal neurons were not stimulated more often than the others
(median 6.5 against 6.5, KS p = 0.994), and amphid neurons were not observed more often (median
27.5 against 32.0, KS p = 0.518; with the NeuroPAL count p = 0.673). The stimulation counts rest
on the 1813 stimulations whose targeted ROI the later version identifies with a head neuron, a subset of the stimulations behind the atlas: 23 neurons
(2 pharyngeal, 2 amphid) have no counted stimulation, yet the atlas gives them 46 out-edges.

## fig5a-signaling-rich-club

> The signaling network has a dual rich club, with 16 neurons of degree 25 or more and a more
> exclusive tier of five neurons of degree 40 or more, at local maxima of the normalized rich
> club coefficient.

Degree is in-degree plus out-degree; phi(k) over nodes of degree > k (richclub.py), normalized
by its mean over 1000 degree-preserving rewired networks (Methods); the first 100 (legend) and
500 (code) of the same draws are variants. The rewiring stands in for graph-tool
`random_rewire(g, "configuration")` (see the helper). Tiers (a) degree > 25 and > 40, (b)
degree >= 25 and >= 40. A local maximum is a k whose normalized value is at least that of both
neighbours and above at least one value within two steps (plateaus of equal values count once,
at their first k).

In [ ]:
deg = nw.total_degree(A)
degB = nw.total_degree(B)


def rich_club(adj, claim_id, n_null=1000, offset=0):
    rng_ = np.random.default_rng(seed_of(claim_id) + offset)
    phi = nw.rich_club_phi(adj)
    null = np.array([nw.rich_club_phi(nw.rewire_configuration(adj, rng_), len(phi))
                     for _ in range(n_null)])
    return phi, null


def local_maxima(x):
    out = []
    for k in range(1, len(x) - 1):
        if not np.isfinite(x[k]) or x[k] == x[k - 1]:
            continue
        nb = x[max(0, k - 2):k + 3]
        if x[k] >= np.nanmax([x[k - 1], x[k + 1]]) and np.nanmin(nb) < x[k]:
            # walk over a plateau to its end before comparing with the next distinct value
            e = k
            while e + 1 < len(x) and x[e + 1] == x[k]:
                e += 1
            if e + 1 >= len(x) or not np.isfinite(x[e + 1]) or x[e + 1] < x[k]:
                out.append(k)
    return out

In [ ]:
cid = "fig5a-signaling-rich-club"
t = time.time()
phi, null = rich_club(A, cid)
print(f"1000 rewired networks in {time.time() - t:.0f} s")
ks = np.arange(len(phi))
table = pd.DataFrame({"k": ks, "N_k": [(deg > k).sum() for k in ks], "phi": phi})
for n_ in (1000, 500, 100):
    table[f"norm_{n_}"] = phi / np.nanmean(null[:n_], axis=0)
table["null_mean"] = np.nanmean(null, axis=0)
table["null_sd"] = np.nanstd(null, axis=0)
table["above_1sd"] = table.phi > table.null_mean + table.null_sd
print(table.round(3).to_string(index=False))
for n_ in (1000, 500, 100):
    print(f"local maxima of the normalized coefficient ({n_} nulls):",
          local_maxima(table[f"norm_{n_}"].to_numpy()))
defined = table[(table.k >= 3) & table.phi.notna() & table.null_mean.notna()]
print("phi minus the null mean, in null SDs, k = 3 to 6:",
      ((table.phi - table.null_mean) / table.null_sd).iloc[3:7].round(2).tolist(),
      "; phi minus null mean at k = 4:", f"{table.phi[4] - table.null_mean[4]:.2e}")
print("k >= 3 with phi above null mean + 1 SD:", int(defined.above_1sd.sum()), "of", len(defined),
      "; first k where it fails:", defined[~defined.above_1sd].k.tolist()[:5])
tier = {}
for rd, op in (("a", np.greater), ("b", np.greater_equal)):
    t1 = sorted(np.array(names)[op(deg, 25)])
    t2 = sorted(np.array(names)[op(deg, 40)])
    tier[rd] = (t1, t2)
    print(f"reading ({rd}): first tier {len(t1)} (expected 16 +/- 1.6), second tier {len(t2)} "
          f"(expected 5 +/- 0.5)")
    print("   ", t1)
    print("    second tier:", t2)
print("reading (a) first tier equals the code's list:", set(tier["a"][0]) == set(nw.CODE_RICH_CLUB))
rows = []
for off in OFFSETS[1:]:
    ph_, nu_ = rich_club(A, cid, offset=off)
    nm = ph_ / np.nanmean(nu_, axis=0)
    mx = local_maxima(nm)
    ab = ph_ > np.nanmean(nu_, axis=0) + np.nanstd(nu_, axis=0)
    ok_k = [k for k in range(3, len(ph_)) if np.isfinite(ph_[k]) and np.isfinite(np.nanmean(nu_[:, k]))]
    rows.append(dict(offset=off, maxima=mx, near25=any(abs(k - 25) <= 2 for k in mx),
                     near40=any(abs(k - 40) <= 2 for k in mx),
                     all_above_1sd=bool(all(ab[k] for k in ok_k))))
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, phi, ".-", label="phi")
ax.errorbar(ks, table.null_mean, yerr=table.null_sd, fmt=".-", label="rewired mean +/- SD")
ax.plot(ks, table.norm_1000, ".-", label="phi normalized")
ax.axhline(1, color="k", lw=0.5)
ax.set_xlabel("k")
ax.legend()
plt.show()
print("plotted k from 0 to", ks.max(), "; phi range", np.nanmin(phi).round(3), "to",
      np.nanmax(phi).round(3), "; normalized range", np.nanmin(table.norm_1000).round(3), "to",
      np.nanmax(table.norm_1000).round(3))

**Result: partially reproduced.** Under reading (a) (degree > 25 and > 40, the legend and the
code) the tiers hold 16 and 5 neurons, the first tier is exactly the code's list, and the
normalized coefficient has local maxima at k = 25 and k = 42, within two of 40. The maxima are k = 3, 23, 25, 29, 42 and 48 with 500 or 1000 rewired
networks, and with 100 also k = 1; under all five seeds they include 25 and 40 or 42. The
coefficient exceeds the null mean by more than one null SD at 47 of the 48 values of k from 3
on, but not at k = 4 (0.88 SD), so the last condition of the criterion fails at one k, under
every seed. Reading (b) (25 or more, 40 or more, as the Results word it) gives 22 and 7 neurons,
outside tolerance. The null comes from a rewiring written in place of graph-tool's.

## fig5c-rich-clubs-disjoint

> The signaling and anatomical rich clubs are nearly disjoint; only AVEL and AVER are in both.

The anatomical rich club is degree > 62 in the anatomical network (richclub.py).

In [ ]:
anat_rc = sorted(np.array(names)[degB > 62])
print(f"anatomical rich club (degree > 62): {len(anat_rc)} neurons (Extended Data Figure 12a gives 15):", anat_rc)
for rd in ("a", "b"):
    print(f"intersection with the signaling first tier, reading ({rd}):",
          sorted(set(tier[rd][0]) & set(anat_rc)))
near = [(names[i], int(degB[i])) for i in np.argsort(-degB) if 55 <= degB[i] <= 70]
print("anatomical degrees near the threshold:", near)
for lab_, src_ in variants.items():
    dv = nw.total_degree(nw.anatomy_adjacency(src_, "any"))
    rc_v = sorted(np.array(names)[dv > 62])
    print(f"variant {lab_}: {len(rc_v)} neurons, intersection with reading (a):",
          sorted(set(tier['a'][0]) & set(rc_v)))

**Result: reproduced.** The anatomical rich club (degree > 62) has 14 neurons, one fewer than
the 15 of the legend of Extended Data Figure 12a, and its intersection with the signaling first tier is exactly AVEL and AVER,
under both readings of the tiers and under both anatomical variants (10 and 8 neurons).

## rich-club-controls (with inputs), the part that can be run

> The near disjointness of the rich clubs persists under the conservative and enforced-density
> controls, and every signaling rich club neuron at q <= 0.18 was in the original signaling
> rich club.

Of the three pairs of control networks, only one can be built here: the original signaling
network against the anatomical network with an average of at least 1.25 contacts over the four
sources (Methods, Network Thresholding Controls; Extended Data Figure 12). The other two need
the signaling network at q < 0.01 and at q <= 0.18, which cannot be built (see the data check).
The conservative anatomical network (at least three contacts in one source) is shown for its
density only.

Rich club thresholds, as the claims file reads them: (a) the original thresholds, degree > 25
for signaling and > 62 for anatomy; (b) each network's own threshold at a local maximum of its
normalized coefficient (1000 rewired networks, as in fig5a-signaling-rich-club). The paper picks
its maxima "by eye" (richclub.py). For (b) the notebook takes, among the local maxima that leave
at least 10 neurons, the one with the largest normalized coefficient; this is an assumption,
checked by showing that it gives back 25 and 62 on the original networks. All local maxima are
printed so that other choices can be seen. The legend of Extended Data Figure 12a (9 of the 15
original anatomical rich club neurons kept, 4 new) is shown for comparison.

In [ ]:
cid = "rich-club-controls"


def club_threshold(adj, offset=0, n_null=1000, min_size=10):
    phi_, null_ = rich_club(adj, cid, n_null=n_null, offset=offset)
    norm_ = phi_ / np.nanmean(null_, axis=0)
    d_ = nw.total_degree(adj)
    cand = [(k, round(float(norm_[k]), 3), int((d_ > k).sum())) for k in local_maxima(norm_)]
    big = [c for c in cand if c[2] >= min_size]
    return max(big, key=lambda c: c[1])[0], cand


B125 = nw.anatomy_adjacency(anat_src, "avg125")
print(f"anatomy, average >= 1.25 contacts: {int(B125.sum())} edges, density {nw.density(B125):.4f} "
      f"(paper 0.033); maximum degree {nw.total_degree(B125).max()}")
for mode in ("three_type", "three_total"):
    Bc = nw.anatomy_adjacency(anat_src, mode)
    print(f"anatomy, conservative ({mode}): {int(Bc.sum())} edges, density {nw.density(Bc):.4f} "
          f"(paper 0.028); its pair needs the signaling network at q < 0.01, not run")
k_sig, cand_sig = club_threshold(A)
k_an, cand_an = club_threshold(B)
print("rule (b) on the original signaling network, local maxima (k, normalized, N_k):", cand_sig,
      "; chosen k:", k_sig)
print("rule (b) on the original anatomical network, local maxima (k, normalized, N_k):", cand_an,
      "; chosen k:", k_an)
sig_club = {"a": set(np.array(names)[deg > 25]), "b": set(np.array(names)[deg > k_sig])}
d125 = nw.total_degree(B125)
rows = []
for off in OFFSETS:
    k125, cand125 = club_threshold(B125, offset=off)
    if off == 0:
        print("1.25-contact anatomy, local maxima (k, normalized, N_k):", cand125)
    for rd, k_ in (("a", 62), ("b", k125)):
        if rd == "a" and off:
            continue
        club = set(np.array(names)[d125 > k_])
        rows.append(dict(seed_offset=off, reading=rd, k=k_, size=len(club),
                         overlap_with_signaling=sorted(club & sig_club[rd]),
                         kept_of_original=f"{len(club & set(anat_rc))} of {len(anat_rc)}",
                         new=sorted(club - set(anat_rc))))
rc_tab = pd.DataFrame(rows)
print(rc_tab.to_string(index=False))
print("1.25-contact anatomical rich club under (b), claim's seed:",
      sorted(np.array(names)[d125 > rc_tab.query("reading == 'b'").k.iloc[0]]))
for lab_, src_ in variants.items():
    Bv = nw.anatomy_adjacency(src_, "avg125")
    kv, _ = club_threshold(Bv)
    dv = nw.total_degree(Bv)
    print(f"variant {lab_}: {int(Bv.sum())} edges, density {nw.density(Bv):.4f}; (a) {int((dv > 62).sum())} "
          f"neurons; (b) k = {kv}, {int((dv > kv).sum())} neurons, overlap with signaling "
          f"{sorted(set(np.array(names)[dv > kv]) & sig_club['b'])}")

**Result: blocked (data missing); the one pair that can be run holds.** The claim needs three
pairs of control networks and the signaling rich club at q <= 0.18; two of the pairs and that
network need q-values the sources do not hold, so the claim stays blocked. The pair that can be
built, the original signaling network against the anatomical network with an average of at
least 1.25 contacts (1161 edges, density 0.0330 against the paper's 0.033), keeps the rich clubs
apart under both readings. Under (a) no neuron of that sparser network has degree above 62 (its
largest degree is 39), so the overlap is empty. Under (b) the rule gives k = 28 under all five
seeds, a club of 13 neurons whose overlap with the signaling rich club (k = 25, which the rule
also gives) is AVEL and AVER only. It keeps 8 of the 14 neurons of the original anatomical rich
club and adds 5 (AIZR, RIAL, RIAR, RIGL, RIGR), where the legend of Extended Data Figure 12a gives
9 of 15 and 4 new. With the White N2U adult the 1.25-contact club under (b) shares only AVER,
and with gap junctions in one direction AVEL and AVER. The rule for (b) is an assumption; it
gives back the paper's thresholds 25 and 62 on the original networks.

## fig5b-degree-correlation

> Signaling and anatomical degree are weakly correlated across neurons (Pearson r = 0.22); the
> highest-degree anatomical neurons tend to have lower signaling degree.

In [ ]:
lr = stats.linregress(degB, deg)
print(f"Pearson r = {lr.rvalue:.4f} (expected 0.22 +/- 0.022), p = {lr.pvalue:.2g}, slope "
      f"{lr.slope:.3f}, n = {N}")
for lab_, src_ in variants.items():
    dv = nw.total_degree(nw.anatomy_adjacency(src_, "any"))
    print(f"variant {lab_}: r = {stats.pearsonr(dv, deg).statistic:.4f}")
top = degB > 62
print(f"signaling degree of the anatomical rich club: median {np.median(deg[top])}, "
      f"others median {np.median(deg[~top])}")
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(degB[~top], deg[~top], s=8, label="other")
ax.scatter(degB[top], deg[top], s=12, c="r", label="anatomical degree > 62")
ax.set_xlabel("anatomical degree")
ax.set_ylabel("signaling degree")
ax.legend()
plt.show()
print("anatomical degree range", degB.min(), "to", degB.max(), "; signaling", deg.min(), "to", deg.max())

**Result: reproduced, resting on the choice of the four EM datasets.** Pearson r = 0.2186
(p = 0.0026) between signaling and anatomical degree, within tolerance. With the White N2U adult
in place of the whole-animal White adult r = 0.169, outside tolerance; with gap junctions in one
direction r = 0.206, inside. The anatomical rich club has a median signaling degree of 9.5
against 10.0 for the other neurons; the second half of the statement is not judged.

## ed10-rich-club-cell-classes

> Signaling rich club neurons are spread fairly equally among sensory, inter- and motor
> neurons, unlike the anatomical rich club, which is mostly interneurons.

In [ ]:
for rd in ("a", "b"):
    c = collections.Counter(cls[[idx[n] for n in tier[rd][0]]])
    print(f"signaling first tier, reading ({rd}), {len(tier[rd][0])} neurons:",
          {k: c.get(k, 0) for k in "SIM"}, "; largest share", round(max(c.values()) / len(tier[rd][0]), 3))
c = collections.Counter(cls[[idx[n] for n in anat_rc]])
print(f"anatomical rich club, {len(anat_rc)} neurons:", {k: c.get(k, 0) for k in "SIM"},
      "; interneuron share", round(c.get("I", 0) / len(anat_rc), 3))
for lab_, src_ in variants.items():
    dv = nw.total_degree(nw.anatomy_adjacency(src_, "any"))
    rc_v = np.array(names)[dv > 62]
    cv = collections.Counter(cls[[idx[n] for n in rc_v]])
    print(f"variant {lab_}: {len(rc_v)} neurons, interneuron share {cv.get('I', 0) / len(rc_v):.3f}")

**Result: reproduced.** The signaling first tier holds 5 sensory, 5 inter- and 6 motor neurons
(largest share 0.375; under reading (b) 11, 5 and 6, largest share 0.5, not more than half).
The anatomical rich club is 12 interneurons of 14 (0.857), and 0.800 and 0.875 under the two
anatomical variants.

## ed9e-degree-observations

> Signaling degree is correlated with the number of times a neuron was observed (Pearson
> r = 0.71), but rich club members have degrees well above what that correlation predicts.

As in degree.py: a least-squares line of total degree on the number of observations, the
residuals, a Gaussian fit to them; a rich club neuron is "well in excess" when its degree is
more than one fitted SD above the line (the claims file's choice). Observations as in
ed9-sampling-not-confound; the NeuroPAL count is a variant.

In [ ]:
for lab_, obs in (("activity ROIs (main)", obs_count), ("NeuroPAL labels (variant)", obs_np)):
    a_, b_ = np.polyfit(obs, deg, 1)
    r_ = stats.pearsonr(obs, deg)
    errors = a_ * obs + b_ - deg
    mu, sd = stats.norm.fit(errors)
    resid = deg - (a_ * obs + b_)
    rc_idx = [idx[n] for n in tier["a"][0]]
    above = resid[rc_idx] > sd
    print(f"{lab_}: r = {r_.statistic:.4f} (expected 0.71 +/- 0.071), p = {r_.pvalue:.2g}; "
          f"fitted SD {sd:.2f}; rich club neurons (reading a) more than 1 SD above the line: "
          f"{above.sum()} of {len(rc_idx)}")
    if lab_.startswith("activity"):
        print(pd.DataFrame({"neuron": tier["a"][0], "observations": obs[rc_idx],
                            "degree": deg[rc_idx], "residual": resid[rc_idx].round(2),
                            "residual/SD": (resid[rc_idx] / sd).round(2)}).to_string(index=False))
        rc_b = [idx[n] for n in tier["b"][0]]
        print(f"reading (b), {len(rc_b)} neurons: more than 1 SD above: {(resid[rc_b] > sd).sum()}")
print("observation counts: range", obs_count.min(), "to", obs_count.max(), "; median",
      np.median(obs_count))
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(obs_count, deg, s=8)
ax.scatter(obs_count[[idx[n] for n in tier["a"][0]]], deg[[idx[n] for n in tier["a"][0]]], s=12,
           c="r", label="signaling rich club (degree > 25)")
ax.set_xlabel("recordings in which the neuron was observed")
ax.set_ylabel("signaling degree")
ax.legend()
plt.show()

**Result: partially reproduced.** Degree and observations correlate with r = 0.702 (expected
0.71). Of the 16 rich club neurons, 10 lie more than one fitted SD (7.34) above the line; AVDL,
AVJL, M3R, RMDDL and RMDVL are within one SD and OLLR is at 1.00 SD. With the NeuroPAL count of
observations r = 0.662 and 11 of 16.

## ave-most-connected-both

> The most connected member of the signaling rich club is also the most connected member of
> the connectome, the bilateral pair AVEL and AVER.

Readings (a) the neuron of highest degree in each network is AVEL or AVER; (b) the class AVE
has the highest summed degree of its left and right neurons among bilateral classes.

In [ ]:
def ranks(d):
    order = np.argsort(-d, kind="stable")
    return pd.DataFrame({"neuron": np.array(names)[order][:6], "degree": d[order][:6]})


print("signaling, top neurons:")
print(ranks(deg).to_string(index=False))
print("anatomy, top neurons:")
print(ranks(degB).to_string(index=False))
for nm, d in (("signaling", deg), ("anatomy", degB)):
    r_ = {n: int((d > d[idx[n]]).sum()) + 1 for n in ("AVEL", "AVER")}
    print(f"{nm}: rank of AVEL {r_['AVEL']}, AVER {r_['AVER']}")
pairs_ = {n[:-1]: (n, n[:-1] + "R") for n in names if n.endswith("L") and n[:-1] + "R" in name_set}
for nm, d in (("signaling", deg), ("anatomy", degB)):
    sums = pd.Series({k: d[idx[l]] + d[idx[r]] for k, (l, r) in pairs_.items()}).sort_values(ascending=False)
    print(f"{nm}, bilateral classes by summed degree:", sums.head(5).to_dict())
for lab_, src_ in variants.items():
    dv = nw.total_degree(nw.anatomy_adjacency(src_, "any"))
    sums = pd.Series({k: dv[idx[l]] + dv[idx[r]] for k, (l, r) in pairs_.items()}).sort_values(ascending=False)
    print(f"variant {lab_}: top neuron {names[int(np.argmax(dv))]} ({dv.max()}); top classes",
          sums.head(3).to_dict())

**Result: reproduced under reading (b), resting on the choice of the four EM datasets.** AVEL
has the highest anatomical degree (92), but the highest signaling degree is AWBL's (52, AVEL 51),
so reading (a) fails. Summed over the bilateral pair, AVE is first in both networks (97 against
94 for AWB; 170 against 166 for AIB). Under both anatomical variants AIB, not AVE, is the first
class in the connectome.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| atlas-dataset-link (link) | partially reproduced: amplitudes agree, 88% of atlas edges measured in the dandiset |
| signaling-network-size | reproduced (reading b, the atlas): 1154 edges, density 0.0328 |
| signaling-clustering-small-world | reproduced: C 0.135, sigma 4.62 |
| anatomy-network-properties (elsewhere) | reproduced, assumed (four EM datasets): 3223 edges, density 0.092, C 0.306, sigma 2.52 |
| triad-distribution | reproduced: chi-square p printed as 0 |
| ed6-signaling-not-feedforward | reproduced: no leg above either null |
| ed6-anatomy-feedforward (elsewhere) | partially reproduced: all legs above the label shuffle, only sensory to motor above the edge swap |
| fig4c-broadcaster-integrator | partially reproduced: amphid holds (p = 0.0046); pharynx out > in, p = 0.073 |
| ed9-sampling-not-confound | reproduced: KS p = 0.994 and 0.518 |
| fig5a-signaling-rich-club | partially reproduced: 16 and 5 neurons, maxima at 25 and 42; one-SD rule fails at k = 4 |
| fig5c-rich-clubs-disjoint | reproduced: AVEL and AVER only |
| fig5b-degree-correlation | reproduced, assumed: r = 0.219 |
| ed10-rich-club-cell-classes | reproduced: 5, 5, 6; anatomy 12 of 14 interneurons |
| ed9e-degree-observations | partially reproduced: r = 0.702; 10 of 16 more than one SD above |
| ave-most-connected-both | reproduced under reading (b), assumed: AVE first by class in both |
| signaling-threshold-controls, bilateral-controls-signaling | blocked, data missing: no q-values other than q < 0.05 |
| rich-club-controls | blocked, data missing; the one pair that can be built (1.25-contact anatomy) shares only AVEL and AVER |
| signaling-six-communities, table1-signaling-membership, anatomy-twenty-communities, more-communities-anatomy-controls, bilateral-same-community-signaling, bilateral-anatomy-stronger, pharynx-community-overlap, seven-enriched-pairings, forced-same-number-ars, pharynx-overlap-controls, fig3b-signaling-cell-type-enrichment, fig3c-signaling-role-enrichment, fig3de-anatomy-enrichment, fig4-community-motif-fractions, fig4d-pharynx-amphid-core, fig4d-anatomy-pharynx-amphid-assortative, fig5d-community-vi-hubs, fig5e-rich-club-core, anatomy-community-motif-fractions, community-vi-most-self-connected, pharynx-less-isolated-signaling | blocked, not runnable: graph-tool (stochastic block model) cannot be installed |

**How the analysis differs from the paper's.** The signaling network is the released atlas of
Randi et al. (2023), not a network computed from the dandiset, because the q-values need control
recordings the dandiset does not hold; it has 1154 edges where the paper has 1151. The atlas
holds only the q < 0.05 network, so the threshold controls cannot be built. The anatomical
network uses the four EM datasets of Randi et al.'s code, which the paper does not name; other
choices change several values. The stochastic block model could not be run, and the paper's printed memberships are not used in its place. The rich club null
uses a rewiring written here in place of graph-tool's. The stimulated-ROI identities come from
the later version of the dandiset.

**What was found about the dataset.** The cited version does not say which ROI each
stimulation targeted (the later version adds it, for about half of the stimulations), and its
green signal holds unmarked interpolated values for 42% of frames and ROIs on average. The
control recordings without stimulation that the atlas's q-values rest on are not deposited, and
the dandiset's labeled stimulations cover 88% of the atlas's edges. Three recordings have raw
imaging but no segmentation. Labels include sideless class names, numbers and free text without
a description. pynwb 4.2.0 cannot build the files. The stimulation times share the signals'
clock, and no recording is deposited twice.

In [ ]:
print(f"notebook runtime {(time.time() - T0) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")